In [1]:
import pandas as pd
import os
import numpy as np

In [2]:
def clean_by_month(df_month):
    df_month['tpep_pickup_datetime'] = pd.to_datetime(df_month['tpep_pickup_datetime'], errors='coerce')
    df_month['tpep_dropoff_datetime'] = pd.to_datetime(df_month['tpep_dropoff_datetime'], errors='coerce')
    #
    numeric_cols = ['passenger_count', 'trip_distance', 'RatecodeID', 'PULocationID', 
                'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 
                'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 
                'congestion_surcharge', 'Airport_fee']

    for col in numeric_cols:
        if col in df_month.columns:
            df_month[col] = pd.to_numeric(df_month[col], errors='coerce')
    #
    if 'store_and_fwd_flag' in df_month.columns:
        df_month['store_and_fwd_flag'] = df_month['store_and_fwd_flag'].fillna('N')
        df_month['store_and_fwd_flag'] = df_month['store_and_fwd_flag'].astype('category')
    
    #
    cols_to_fill_zero = ['tip_amount', 'tolls_amount', 'Airport_fee', 'congestion_surcharge', 
                   'improvement_surcharge', 'extra', 'mta_tax']
    for col in cols_to_fill_zero:
        if col in df_month.columns:
            df_month[col] = df_month[col].fillna(0)

    #
    cols_to_fill_mode = ['passenger_count', 'RatecodeID', 'payment_type', 'PULocationID', 'DOLocationID']
    for col in cols_to_fill_mode:
        if col in df_month.columns and df_month[col].isnull().any():
            mode_val = df_month[col].mode()[0]
            print(f"Điền NaN cho cột '{col}' bằng giá trị mode: {mode_val}")
            df_month[col] = df_month[col].fillna(mode_val)
    #
    int_cols = ['passenger_count', 'RatecodeID', 'PULocationID', 'DOLocationID', 'payment_type']
    for col in int_cols:
        if col in df_month.columns:
            df_month[col] = df_month[col].astype("int32")

    #
    def add_flag(df, condition, flag_name):
        df.loc[condition, 'qa_flags'] = df.loc[condition, 'qa_flags'].apply(lambda x: x + [flag_name])
    
    #
    df_month['qa_flags'] = [[] for _ in range(len(df_month))]

    swap_condition = df_month["tpep_pickup_datetime"] > df_month["tpep_dropoff_datetime"]
    add_flag(df_month, swap_condition, 'FLAG_TIME_SWAPPED_FIXED')
    rows_swapped = swap_condition.sum()
    if rows_swapped > 0:
        print(f"Phát hiện và gắn cờ {rows_swapped} hàng có thời gian bị ngược.")
        
        cols_to_swap = ['tpep_pickup_datetime', 'tpep_dropoff_datetime']
        swapped_cols = ['tpep_dropoff_datetime', 'tpep_pickup_datetime']
        df_month.loc[swap_condition, cols_to_swap] = df_month.loc[swap_condition, swapped_cols].values
    
    #
    df_month['trip_duration_minutes'] = (df_month['tpep_dropoff_datetime'] - df_month['tpep_pickup_datetime']).dt.total_seconds() / 60
    df_month['speed_mph'] = df_month['trip_distance'] / (df_month['trip_duration_minutes'] / 60 + 1e-6)
    df_month['speed_mph'] = df_month['speed_mph'].replace([np.inf, -np.inf], 0) 


    #
    add_flag(df_month, (df_month['trip_duration_minutes'] <= 2) & (df_month['trip_distance'] > 0), 'FLAG_DURATION_TOO_SHORT')
    add_flag(df_month, df_month['trip_duration_minutes'] > 1440, 'FLAG_DURATION_TOO_LONG') 

    #
    add_flag(df_month, df_month['trip_distance'] <= 0, 'FLAG_DISTANCE_ZERO_OR_NEG')
    add_flag(df_month, df_month['trip_distance'] > 200, 'FLAG_DISTANCE_EXTREME') # > 200 dặm
    add_flag(df_month, (df_month["passenger_count"] <= 0) & (df_month['trip_distance'] > 0), 'FLAG_PASSENGER_INVALID')
    add_flag(df_month, df_month["fare_amount"] < 3.0, "FLAG_FARE_AMOUNT_INVALID")

    #
    additive_fee_cols = [
    'extra', 
    'mta_tax', 
    'tip_amount', 
    'tolls_amount', 
    'improvement_surcharge', 
    'congestion_surcharge', 
    'Airport_fee'
    ]


    for col in additive_fee_cols:
        if col in df_month.columns:
            condition = df_month[col] < 0
            if condition.any(): 
                flag_name = f'FLAG_NEGATIVE_{col.upper()}' # Tên cờ động, vd: FLAG_NEGATIVE_TIP_AMOUNT
                add_flag(df_month, condition, flag_name)
                print(f"Đã gắn cờ cho {condition.sum()} hàng có {col} bị âm.")


    #
    key_cols = ['VendorID','tpep_pickup_datetime', 'PULocationID', 'trip_distance', 'total_amount']
    duplicates_mask = df_month.duplicated(subset=key_cols, keep='first')
    add_flag(df_month, duplicates_mask, 'FLAG_DUPLICATE')

    #
    df_month['is_clean'] = df_month['qa_flags'].apply(lambda x: len(x) == 0)
    print(f"-> Hoàn thành. {df_month['is_clean'].sum()} hàng sạch / {len(df_month)} tổng số hàng.")

    return df_month

In [3]:
path = r"C:\Users\APC\2526-LTXLDL-Project-1.5\raw"

## Đảm bảo thư mục tồn tại

In [4]:

os.makedirs('reports', exist_ok=True)
os.makedirs('processed/clean_data_monthly', exist_ok=True) # Nơi lưu 12 tệp sạch
os.makedirs('processed/full_data_monthly', exist_ok=True) # Nơi lưu 12 tệp đầy đủ

In [5]:
list_of_qa_reports = [] 
for file in os.listdir(path):
    if file.endswith(".parquet"):
        filepath = os.path.join(path, file)
        print(f"\nĐang đọc tệp: {file}")
        temp_raw = pd.read_parquet(filepath)

        if "airport_fee" in temp_raw.columns:
            temp_raw = temp_raw.rename(columns={"airport_fee" : "Airport_fee"}) # Xử lý vì tháng 1 khác biệt
        
        df_full_flagged = clean_by_month(temp_raw)
        
        all_flags = df_full_flagged.explode('qa_flags')['qa_flags'].dropna()
        qa_summary_month = pd.DataFrame(all_flags.value_counts())
        qa_summary_month.columns = ['SoLuongViPham']

        month_name = file.replace('yellow_tripdata_', '').replace('.parquet', '')
        qa_summary_month['Month'] = month_name
        list_of_qa_reports.append(qa_summary_month)

        # LỌC và LƯU TỆP SẠCH cho 1 tháng 
        df_clean_month = df_full_flagged[df_full_flagged['is_clean'] == True]
        # Lấy cột gốc
        cols_to_drop = ['qa_flags','is_clean']
        original_cols = [col for col in df_full_flagged.columns if col not in cols_to_drop]
        df_clean_final = df_clean_month[original_cols]


        # LƯU RA TỆP .parquet CỦA THÁNG ĐÓ
        clean_output_path = f'processed/clean_data_monthly/{month_name}_clean.parquet'
        df_clean_final.to_parquet(clean_output_path, index=False, compression='snappy')
        print(f"ĐÃ LƯU TỆP SẠCH: {clean_output_path}")

        # full_output_path = f'processed/full_data_monthly/{month_name}_full.parquet'
        # df_full_flagged.to_parquet(full_output_path, index=False, compression='snappy')

        # GIẢI PHÓNG BỘ NHỚ
        del temp_raw, df_full_flagged, df_clean_month, df_clean_final
        print(f"Đã giải phóng bộ nhớ của tháng {month_name}")

print("\n=====================Xong gòi========================")



Đang đọc tệp: yellow_tripdata_2023-01.parquet
Điền NaN cho cột 'passenger_count' bằng giá trị mode: 1.0
Điền NaN cho cột 'RatecodeID' bằng giá trị mode: 1.0
Phát hiện và gắn cờ 3 hàng có thời gian bị ngược.
Đã gắn cờ cho 12407 hàng có extra bị âm.
Đã gắn cờ cho 24501 hàng có mta_tax bị âm.
Đã gắn cờ cho 225 hàng có tip_amount bị âm.
Đã gắn cờ cho 1377 hàng có tolls_amount bị âm.
Đã gắn cờ cho 25153 hàng có improvement_surcharge bị âm.
Đã gắn cờ cho 19718 hàng có congestion_surcharge bị âm.
Đã gắn cờ cho 3607 hàng có Airport_fee bị âm.
-> Hoàn thành. 2917483 hàng sạch / 3066766 tổng số hàng.
ĐÃ LƯU TỆP SẠCH: processed/clean_data_monthly/2023-01_clean.parquet
Đã giải phóng bộ nhớ của tháng 2023-01

Đang đọc tệp: yellow_tripdata_2023-02.parquet
Điền NaN cho cột 'passenger_count' bằng giá trị mode: 1.0
Điền NaN cho cột 'RatecodeID' bằng giá trị mode: 1.0
Phát hiện và gắn cờ 276 hàng có thời gian bị ngược.
Đã gắn cờ cho 12300 hàng có extra bị âm.
Đã gắn cờ cho 24249 hàng có mta_tax bị âm.


In [6]:
# --- GỘP BÁO CÁO QA ---
if not list_of_qa_reports:
    print("Không có báo cáo QA nào để gộp.")
else:
    # Gộp 12 báo cáo
    df_qa_annual = pd.concat(list_of_qa_reports) #Gộp

    # Xử lý lại bảng báo cáo tổng
    df_qa_annual.reset_index(inplace=True) # Chuyển index thành cột (tên cột là 'qa_flags')
    
    # Đổi tên cột 'qa_flags'
    df_qa_annual = df_qa_annual.rename(columns={'qa_flags': 'QuyTacQA'})
    
    # Tệp này cho bạn biết lỗi nào xảy ra ở tháng nào
    qa_detailed_path = 'reports/qa_summary_DETAILED_BY_MONTH.csv'
    df_qa_annual.to_csv(qa_detailed_path, encoding='utf-8-sig', index=False)
    print(f"ĐÃ LƯU BÁO CÁO CHI TIẾT (cho Mục 5, 7) vào: {qa_detailed_path}")
    #
    qa_summary_final = df_qa_annual.groupby('QuyTacQA')['SoLuongViPham'].sum().sort_values(ascending=False)
    qa_summary_final = pd.DataFrame(qa_summary_final)
    
    # Tính toán lại tỷ lệ và quyết định
    total_rows_annual = qa_summary_final['SoLuongViPham'].sum() # TỔNG SỐ LỖI
    if total_rows_annual == 0:
        qa_summary_final['TyLeViPham (%)'] = 0.0
    else:
        qa_summary_final['TyLeViPham (%)'] = (qa_summary_final['SoLuongViPham'] / total_rows_annual) * 100
    
    decisions = []
    for flag in qa_summary_final.index:
        if 'FIXED' in str(flag): decisions.append('Sửa & Giữ cho KPI')
        else: decisions.append('Gắn cờ & Loại khỏi thống kê KPI')
    qa_summary_final['QuyetDinhXuLy'] = decisions
    
    # Lưu tệp summary
    qa_summary_path = 'reports/qa_summary_ANNUAL.csv'
    qa_summary_final.to_csv(qa_summary_path, encoding='utf-8-sig')
    
    print(f"ĐÃ LƯU BÁO CÁO QA TỔNG CẢ NĂM: {qa_summary_path}")
    print(qa_summary_final)
    
print("\n--- Xong ---")

ĐÃ LƯU BÁO CÁO CHI TIẾT (cho Mục 5, 7) vào: reports/qa_summary_DETAILED_BY_MONTH.csv
ĐÃ LƯU BÁO CÁO QA TỔNG CẢ NĂM: reports/qa_summary_ANNUAL.csv
                                     SoLuongViPham  TyLeViPham (%)  \
QuyTacQA                                                             
FLAG_DISTANCE_ZERO_OR_NEG                   773457       22.215811   
FLAG_PASSENGER_INVALID                      567802       16.308834   
FLAG_DURATION_TOO_SHORT                     419163       12.039513   
FLAG_FARE_AMOUNT_INVALID                    404186       11.609333   
FLAG_NEGATIVE_IMPROVEMENT_SURCHARGE         376673       10.819084   
FLAG_NEGATIVE_MTA_TAX                       365790       10.506494   
FLAG_NEGATIVE_CONGESTION_SURCHARGE          303052        8.704486   
FLAG_NEGATIVE_EXTRA                         190753        5.478950   
FLAG_NEGATIVE_AIRPORT_FEE                    50293        1.444553   
FLAG_NEGATIVE_TOLLS_AMOUNT                   24764        0.711290   
FLAG_TIME_SWAP